In [1]:
import os
import json
import argparse
import matplotlib.pyplot as plt
import numpy as np
from typing import Dict, List, Optional


def load_json(path: str) -> Dict:
    """Load JSON file."""
    with open(path, "r") as f:
        return json.load(f)

In [2]:
def plot_all_ab_evaluation(models_results: Dict[str, Dict], output_path: str, title: str = "A/B Evaluation"):
    """
    Plot A/B evaluation results for multiple models on a single plot.
    
    Args:
        models_results: Dictionary mapping model names to their results dictionaries
        output_path: Path to save the plot
        title: Plot title
    """
    # Define colors for each model
    colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#6A994E', '#BC4749']
    markers = ['o', 's', '^', 'D', 'v', 'p']
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Behavior score plot
    ax1 = axes[0]
    # Matching probability plot
    ax2 = axes[1]
    
    for idx, (model_name, results) in enumerate(models_results.items()):
        multipliers = sorted([float(m) for m in results.keys()])
        behavior_scores = [results[str(m) if str(m) in results else m]["behavior_score"] for m in multipliers]
        matching_probs = [results[str(m) if str(m) in results else m]["matching_prob"] for m in multipliers]
        not_matching_probs = [results[str(m) if str(m) in results else m]["not_matching_prob"] for m in multipliers]
        
        color = colors[idx % len(colors)]
        marker = markers[idx % len(markers)]
        
        # Plot behavior scores
        ax1.plot(multipliers, behavior_scores, marker=marker, linestyle='-', 
                markersize=8, linewidth=2, color=color, label=model_name)
        
        # Plot matching probabilities
        ax2.plot(multipliers, matching_probs, marker=marker, linestyle='-', 
                markersize=8, linewidth=2, color=color, label=f'{model_name} (Match)')
        ax2.plot(multipliers, not_matching_probs, marker=marker, linestyle='--', 
                markersize=6, linewidth=1.5, color=color, alpha=0.6, label=f'{model_name} (No Match)')
    
    # Format behavior score plot
    ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax1.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    ax1.set_xlabel('Steering Multiplier', fontsize=12)
    ax1.set_ylabel('Behavior Score (log odds)', fontsize=12)
    ax1.set_title('Behavior Score vs Steering', fontsize=14)
    ax1.grid(True, alpha=0.3)
    ax1.legend(fontsize=9, loc='best')
    
    # Format matching probability plot
    ax2.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random chance')
    ax2.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    ax2.set_xlabel('Steering Multiplier', fontsize=12)
    ax2.set_ylabel('Probability', fontsize=12)
    ax2.set_title('Matching Probability vs Steering', fontsize=14)
    ax2.set_ylim(0, 1)
    ax2.grid(True, alpha=0.3)
    ax2.legend(fontsize=8, loc='best', ncol=2)
    
    plt.suptitle(title, fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: {output_path}")

models = ['gemma-7b-it', 'Llama-2-7b-chat-hf', 'Llama-2-13b-chat-hf', 'Meta-Llama-3-8B-Instruct']

In [4]:
models_results = {
    'gemma-7b-it': load_json('pipeline/runs/gemma-7b-it/survival-instinct/evaluations/ab_evaluation.json'),
    'Llama-2-7b-chat-hf': load_json('pipeline/runs/Llama-2-7b-chat-hf/survival-instinct/evaluations/ab_evaluation.json'),
    'Llama-2-13b-chat-hf': load_json('pipeline/runs/Llama-2-13b-chat-hf/survival-instinct/evaluations/ab_evaluation.json'),
    'Meta-Llama-3-8B-Instruct': load_json('pipeline/runs/Meta-Llama-3-8B-Instruct/survival-instinct/evaluations/ab_evaluation.json')
}

plot_all_ab_evaluation(models_results, 'custom_plots/ab_output.png', 'A/B Evaluation - All Models')

Saved: custom_plots/ab_output.png
